# 01 - Extracción y preparación de datos

**Reto 21 - Detección de pases (Kaggle: DFL - Bundesliga Data Shootout)**

Toma `data/raw/train_annotations.csv` (mirror HuggingFace) + los videos del mirror de
Kaggle y produce `data/processed/`: `events.csv`, `plays.csv`, `video_meta.csv`,
`clip_index.csv`, `frames_index.csv` y los JPG en `frames/`.

Fuentes, cómo descargarlas y diccionario de datos: `README.md` y `data/processed/README.md`.
Secciones 2-5 corren solo con las anotaciones; 6-8 necesitan los videos + OpenCV.

## 1. Configuración

In [1]:
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd

# --- Rutas ---------------------------------------------------------------
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw"
OUT_DIR = PROJECT_DIR / "data" / "processed"
FRAMES_DIR = OUT_DIR / "frames"

ANNOTATIONS_CSV = RAW_DIR / "train_annotations.csv"
# Carpeta con las subcarpetas por video (mirror de Kaggle). Ajustar si cambia.
VIDEOS_ROOT = RAW_DIR / "DFL Bundesliga Data Shootout" / "train"

OUT_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

# --- Parámetros ---------------------------------------------------------
CLIP_SECONDS = 30.0        # duración asumida de cada clip del mirror
EVENT_WINDOW_S = 2.0       # ventana +/- alrededor de cada evento puntual
EVENT_WINDOW_STEP_S = 0.5  # separación entre frames dentro de la ventana
BG_EVERY_S = 10.0          # 1 frame de contexto cada N segundos
MAX_VIDEOS_FRAMES = 3      # None = todos; p.ej. 3 para una corrida rápida

print("PROJECT_DIR :", PROJECT_DIR)
print("annotations :", ANNOTATIONS_CSV, "(existe:", ANNOTATIONS_CSV.exists(), ")")
print("videos root :", VIDEOS_ROOT, "(existe:", VIDEOS_ROOT.exists(), ")")

PROJECT_DIR : /Users/nilsmurallesmorales/universidad/octavosemestre/data/p2
annotations : /Users/nilsmurallesmorales/universidad/octavosemestre/data/p2/data/raw/train_annotations.csv (existe: True )
videos root : /Users/nilsmurallesmorales/universidad/octavosemestre/data/p2/data/raw/DFL Bundesliga Data Shootout/train (existe: True )


## 2. Anotaciones crudas

`event` ∈ {`start`, `end` (marcadores de jugada), `play`, `throwin`, `challenge` (eventos puntuales)}.
`time` en segundos absolutos; `event_attributes` es una lista serializada.

In [2]:
raw = pd.read_csv(ANNOTATIONS_CSV)
print("shape:", raw.shape)
print("\ntipos:\n", raw.dtypes, sep="")
print("\nvalores de 'event':\n", raw["event"].value_counts(dropna=False), sep="")
print("\nvideos anotados:", raw["video_id"].nunique())
raw.head(12)

shape: (11218, 4)

tipos:
video_id                str
time                float64
event                   str
event_attributes        str
dtype: object

valores de 'event':
event
play         3586
start        3418
end          3418
challenge     624
throwin       172
Name: count, dtype: int64

videos anotados: 12


,video_id,time,event,event_attributes
0,1606b0e6_0,200.265822,start,NaN
1,1606b0e6_0,201.150000,challenge,['ball_action_forced']
2,1606b0e6_0,202.765822,end,NaN
3,1606b0e6_0,210.124111,start,NaN
4,1606b0e6_0,210.870000,challenge,['opponent_dispossessed']
5,1606b0e6_0,212.624111,end,NaN
6,1606b0e6_0,217.850213,start,NaN
7,1606b0e6_0,219.230000,throwin,['pass']
8,1606b0e6_0,220.350213,end,NaN
9,1606b0e6_0,223.930850,start,NaN


In [3]:
# Chequeo de calidad ANTES de tocar nada (evidencia para la sección de limpieza)
print("Nulos por columna:\n", raw.isna().sum(), sep="")
print("\nFilas duplicadas exactas:", raw.duplicated().sum())
print("time < 0        :", (raw["time"] < 0).sum())
print("time == NaN     :", raw["time"].isna().sum())
print("\nevent_attributes vacío por tipo de evento:")
print(raw.assign(attr_vacio=raw["event_attributes"].isna())
         .groupby("event")["attr_vacio"].mean().round(3))

Nulos por columna:
video_id               0
time                   0
event                  0
event_attributes    6836
dtype: int64

Filas duplicadas exactas: 0
time < 0        : 0
time == NaN     : 0

event_attributes vacío por tipo de evento:
event
challenge    0.0
end          1.0
play         0.0
start        1.0
throwin      0.0
Name: attr_vacio, dtype: float64


## 3. Normalizar eventos → `events.csv`

Explota `event_attributes` en `attr_1`/`attr_2`, marca `is_point_event` y calcula a qué
clip de 30 s del mirror cae cada evento.

In [4]:
def parse_attrs(val):
    """\"['pass', 'openplay']\" -> ['pass', 'openplay']; NaN/'' -> []."""
    if isinstance(val, list):
        return val
    if pd.isna(val) or str(val).strip() in ("", "[]"):
        return []
    try:
        parsed = ast.literal_eval(str(val))
        return list(parsed) if isinstance(parsed, (list, tuple)) else [parsed]
    except (ValueError, SyntaxError):
        return [s.strip(" '\"") for s in str(val).strip("[]").split(",") if s.strip()]


POINT_EVENTS = {"play", "throwin", "challenge"}

events = raw.rename(columns={"time": "time_s"}).copy()
attrs = events["event_attributes"].apply(parse_attrs)
max_attrs = int(attrs.str.len().max() or 0)
for i in range(max_attrs):
    events[f"attr_{i + 1}"] = attrs.apply(lambda a, i=i: a[i] if len(a) > i else np.nan)

events["is_point_event"] = events["event"].isin(POINT_EVENTS)
events["clip_num"] = (events["time_s"] // CLIP_SECONDS).astype(int) + 1
events["time_in_clip_s"] = (events["time_s"] % CLIP_SECONDS).round(3)
events = events.sort_values(["video_id", "time_s"]).reset_index(drop=True)

print("nº máximo de atributos por evento:", max_attrs)
events.head(12)

nº máximo de atributos por evento: 2


,video_id,time_s,event,event_attributes,attr_1,attr_2,is_point_event,clip_num,time_in_clip_s
0,1606b0e6_0,200.265822,start,NaN,NaN,NaN,False,7,20.266
1,1606b0e6_0,201.150000,challenge,['ball_action_forced'],ball_action_forced,NaN,True,7,21.150
2,1606b0e6_0,202.765822,end,NaN,NaN,NaN,False,7,22.766
3,1606b0e6_0,210.124111,start,NaN,NaN,NaN,False,8,0.124
4,1606b0e6_0,210.870000,challenge,['opponent_dispossessed'],opponent_dispossessed,NaN,True,8,0.870
5,1606b0e6_0,212.624111,end,NaN,NaN,NaN,False,8,2.624
6,1606b0e6_0,217.850213,start,NaN,NaN,NaN,False,8,7.850
7,1606b0e6_0,219.230000,throwin,['pass'],pass,NaN,True,8,9.230
8,1606b0e6_0,220.350213,end,NaN,NaN,NaN,False,8,10.350
9,1606b0e6_0,223.930850,start,NaN,NaN,NaN,False,8,13.931


## 4. Intervalos de jugada → `plays.csv`

Empareja `start`/`end` por video; los marcadores sin pareja van a `plays_unmatched.csv`.

In [5]:
rows, unmatched = [], []
for vid, g in events[events["event"].isin(["start", "end"])].groupby("video_id"):
    open_start = None
    for _, r in g.sort_values("time_s").iterrows():
        if r["event"] == "start":
            if open_start is not None:
                unmatched.append({"video_id": vid, "time_s": open_start, "kind": "start_sin_end"})
            open_start = r["time_s"]
        else:
            if open_start is None:
                unmatched.append({"video_id": vid, "time_s": r["time_s"], "kind": "end_sin_start"})
            else:
                rows.append({"video_id": vid, "start_s": open_start, "end_s": r["time_s"],
                             "duration_s": round(r["time_s"] - open_start, 3)})
                open_start = None
    if open_start is not None:
        unmatched.append({"video_id": vid, "time_s": open_start, "kind": "start_sin_end"})

plays = pd.DataFrame(rows)
unmatched_df = pd.DataFrame(unmatched)

# nº de eventos puntuales dentro de cada jugada
pe = events[events["is_point_event"]]
plays["n_point_events"] = plays.apply(
    lambda r: int(((pe["video_id"] == r["video_id"]) &
                   pe["time_s"].between(r["start_s"], r["end_s"])).sum()), axis=1)

print("jugadas emparejadas:", len(plays), "| marcadores sueltos:", len(unmatched_df))
display(unmatched_df.head())
plays.head()

jugadas emparejadas: 3418 | marcadores sueltos: 0


""


,video_id,start_s,end_s,duration_s,n_point_events
0,1606b0e6_0,200.265822,202.765822,2.5,1
1,1606b0e6_0,210.124111,212.624111,2.5,1
2,1606b0e6_0,217.850213,220.350213,2.5,1
3,1606b0e6_0,223.930850,226.430850,2.5,1
4,1606b0e6_0,228.955367,231.455367,2.5,1


## 5. Guardar tablas de anotaciones

In [6]:
events.to_csv(OUT_DIR / "events.csv", index=False)
plays.to_csv(OUT_DIR / "plays.csv", index=False)
if not unmatched_df.empty:
    unmatched_df.to_csv(OUT_DIR / "plays_unmatched.csv", index=False)

for f in sorted(OUT_DIR.glob("*.csv")):
    print(f.name, pd.read_csv(f).shape)

clip_index.csv (425, 5)
events.csv (11218, 9)
plays.csv (3418, 5)
video_meta.csv (12, 8)


## 6. Índice de clips y metadata de video

Recorre `VIDEOS_ROOT`. Cada subcarpeta es un video (puede llevar un prefijo de una letra:
`A1606b0e6_0` → `1606b0e6_0`); cada `.mp4` es un clip numerado `(N)` que cubre
`[(N-1)·30, N·30)` s. Requiere OpenCV para fps/resolución.

In [7]:
try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False
    print("cv2 no disponible -> se omiten fps/resolución (se usa duración teórica).")

KNOWN_VIDS = set(events["video_id"].unique())
CLIP_NUM_RE = re.compile(r"\((\d+)\)")


def folder_to_video_id(name):
    if name in KNOWN_VIDS:
        return name
    if name[1:] in KNOWN_VIDS:   # prefijo de una letra
        return name[1:]
    return name


clip_rows = []
if VIDEOS_ROOT.exists():
    for folder in sorted(p for p in VIDEOS_ROOT.iterdir() if p.is_dir()):
        vid = folder_to_video_id(folder.name)
        for mp4 in folder.glob("*.mp4"):
            m = CLIP_NUM_RE.search(mp4.name)
            if not m:
                continue
            n = int(m.group(1))
            clip_rows.append({
                "video_id": vid,
                "clip_num": n,
                "start_s": (n - 1) * CLIP_SECONDS,
                "end_s": n * CLIP_SECONDS,
                "path": str(mp4.relative_to(PROJECT_DIR)),
            })

clip_index = pd.DataFrame(clip_rows).sort_values(["video_id", "clip_num"]).reset_index(drop=True)
print(f"{len(clip_index)} clips en {clip_index['video_id'].nunique()} videos")
clip_index.head()

425 clips en 5 videos


,video_id,clip_num,start_s,end_s,path
0,1606b0e6_0,1,0.0,30.0,data/raw/DFL Bundesliga Data Shootout/train/A1...
1,1606b0e6_0,2,30.0,60.0,data/raw/DFL Bundesliga Data Shootout/train/A1...
2,1606b0e6_0,3,60.0,90.0,data/raw/DFL Bundesliga Data Shootout/train/A1...
3,1606b0e6_0,4,90.0,120.0,data/raw/DFL Bundesliga Data Shootout/train/A1...
4,1606b0e6_0,5,120.0,150.0,data/raw/DFL Bundesliga Data Shootout/train/A1...


In [8]:
def probe(path):
    cap = cv2.VideoCapture(str(PROJECT_DIR / path))
    if not cap.isOpened():
        return {}
    fps = cap.get(cv2.CAP_PROP_FPS) or np.nan
    nf = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    w, h = int(cap.get(3) or 0), int(cap.get(4) or 0)
    cap.release()
    return {"fps": round(fps, 3), "clip_frames": nf,
            "clip_dur_s": round(nf / fps, 2) if fps else np.nan,
            "width": w, "height": h,
            "aspect_ratio": round(w / h, 4) if h else np.nan}


ev_all = events.groupby("video_id").size().to_dict()
ev_pt = events[events["is_point_event"]].groupby(["video_id", "event"]).size().unstack(fill_value=0)

meta_rows = []
for vid in sorted(KNOWN_VIDS):
    clips = clip_index[clip_index["video_id"] == vid]
    row = {
        "video_id": vid,
        "has_video": len(clips) > 0,
        "n_clips": len(clips),
        "n_events": int(ev_all.get(vid, 0)),
        "n_play": int(ev_pt.loc[vid, "play"]) if vid in ev_pt.index else 0,
        "n_throwin": int(ev_pt.loc[vid, "throwin"]) if vid in ev_pt.index else 0,
        "n_challenge": int(ev_pt.loc[vid, "challenge"]) if vid in ev_pt.index else 0,
        "annot_time_max_s": round(events.loc[events["video_id"] == vid, "time_s"].max(), 1),
    }
    if HAS_CV2 and len(clips):
        row.update(probe(clips.iloc[0]["path"]))
        row["approx_video_dur_s"] = len(clips) * CLIP_SECONDS
    meta_rows.append(row)

video_meta = pd.DataFrame(meta_rows)

# frame absoluto del evento (si tenemos fps)
fps_map = video_meta.set_index("video_id")["fps"].to_dict() if "fps" in video_meta else {}
events["fps"] = events["video_id"].map(fps_map)
events["frame_abs"] = np.floor(events["time_s"] * events["fps"]).astype("Int64")
events.to_csv(OUT_DIR / "events.csv", index=False)

clip_index.to_csv(OUT_DIR / "clip_index.csv", index=False)
video_meta.to_csv(OUT_DIR / "video_meta.csv", index=False)
video_meta

,video_id,has_video,n_clips,n_events,n_play,n_throwin,n_challenge,annot_time_max_s,fps,clip_frames,clip_dur_s,width,height,aspect_ratio,approx_video_dur_s
0,1606b0e6_0,True,76,1000,319,21,56,3014.7,25.0,750.0,30.0,1920.0,1080.0,1.7778,2280.0
1,1606b0e6_1,True,92,1249,434,13,60,3387.7,25.0,750.0,30.0,1920.0,1080.0,1.7778,2760.0
2,35bd9041_0,True,83,1075,355,8,48,3061.9,25.0,750.0,30.0,1920.0,1080.0,1.7778,2490.0
3,35bd9041_1,True,93,933,290,12,57,3477.7,25.0,750.0,30.0,1920.0,1080.0,1.7778,2790.0
4,3c993bd2_0,True,81,1042,347,14,53,2999.0,25.0,750.0,30.0,1920.0,1080.0,1.7778,2430.0
5,3c993bd2_1,False,0,966,289,17,70,3472.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,407c5a9e_1,False,0,858,283,11,56,3445.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,4ffd5986_0,False,0,792,235,10,57,3099.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9a97dae4_1,False,0,737,237,6,48,2999.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,cfbe2e94_0,False,0,823,241,29,35,3216.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 7. Muestreo de frames → `data/processed/frames/`

1 frame de contexto cada `BG_EVERY_S` s + ventana `±EVENT_WINDOW_S` (paso `EVENT_WINDOW_STEP_S`)
en cada evento puntual. Nombre: `{video_id}__t{seg}s__{label}.jpg`. Parte lenta:
`MAX_VIDEOS_FRAMES` limita cuántos videos se procesan.

In [9]:
def grab_frame(cap, t_local, fps):
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(t_local * fps)))
    ok, frame = cap.read()
    return frame if ok else None


def sample_video(vid):
    clips = clip_index[clip_index["video_id"] == vid].set_index("clip_num")
    if clips.empty:
        return 0
    row = video_meta.loc[video_meta["video_id"] == vid].iloc[0]
    fps = float(row["fps"]) if "fps" in row and not pd.isna(row["fps"]) else 25.0
    dur = clips.index.max() * CLIP_SECONDS
    out_dir = FRAMES_DIR / vid
    out_dir.mkdir(parents=True, exist_ok=True)

    targets = {round(t, 1): "bg" for t in np.arange(0, dur, BG_EVERY_S)}
    for _, e in events[(events["video_id"] == vid) & events["is_point_event"]].iterrows():
        for w in np.arange(-EVENT_WINDOW_S, EVENT_WINDOW_S + 1e-9, EVENT_WINDOW_STEP_S):
            ts = round(e["time_s"] + w, 1)
            if 0 <= ts < dur:
                targets[ts] = e["event"]

    saved, open_cap, open_n = 0, None, None
    for ts, label in sorted(targets.items()):
        n = int(ts // CLIP_SECONDS) + 1
        if n not in clips.index:
            continue
        if n != open_n:
            if open_cap is not None:
                open_cap.release()
            open_cap = cv2.VideoCapture(str(PROJECT_DIR / clips.loc[n, "path"]))
            open_n = n
        frame = grab_frame(open_cap, ts - (n - 1) * CLIP_SECONDS, fps)
        if frame is None:
            continue
        cv2.imwrite(str(out_dir / f"{vid}__t{ts:07.1f}s__{label}.jpg"),
                    frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
        saved += 1
    if open_cap is not None:
        open_cap.release()
    return saved


if HAS_CV2 and not clip_index.empty:
    try:
        from tqdm.auto import tqdm
    except ImportError:
        def tqdm(x, **k):
            return x
    vids = sorted(clip_index["video_id"].unique())
    if MAX_VIDEOS_FRAMES:
        vids = vids[:MAX_VIDEOS_FRAMES]
    total = sum(sample_video(v) for v in tqdm(vids, desc="videos"))
    print(f"\n{total} frames en {FRAMES_DIR}")
else:
    print("Sin cv2 o sin clips: muestreo omitido.")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


videos:   0%|          | 0/3 [00:00<?, ?it/s]

videos:  33%|███▎      | 1/3 [02:40<05:20, 160.20s/it]

[ WARN:0@1584.057] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 152337.701567 ms
[ WARN:0@1584.058] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 152337.894076 ms
[mov,mp4,m4a,3gp,3g2,mj2 @ 0xae20c0780] stream 0, offset 0xf15e72: partial file
[mov,mp4,m4a,3gp,3g2,mj2 @ 0xae20c0780] stream 0, offset 0xf1770b: partial file


videos:  67%|██████▋   | 2/3 [26:31<15:07, 907.66s/it]

[ WARN:0@3290.223] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 352725.532902 ms
[h264 @ 0xae24f4e00] Invalid NAL unit size (9752 > 1457).
[h264 @ 0xae24f4e00] missing picture in access unit with size 1461
[h264 @ 0xae24f7100] Invalid NAL unit size (9752 > 1457).
[h264 @ 0xae24f7100] Error splitting the input into NAL units.


[ WARN:0@3987.126] global cap_ffmpeg_impl.hpp:453 _opencv_ffmpeg_interrupt_callback Stream timeout triggered after 689985.662687 ms
[h264 @ 0xae24f4e00] Invalid NAL unit size (105807 > 16667).
[h264 @ 0xae24f4e00] missing picture in access unit with size 16671
[h264 @ 0xae24f6d80] Invalid NAL unit size (105807 > 16667).
[h264 @ 0xae24f6d80] Error splitting the input into NAL units.
[h264 @ 0xae24f5f80] reference picture missing during reorder
[h264 @ 0xae24f5f80] Missing reference picture, default is 65560
[h264 @ 0xae24f4700] reference picture missing during reorder
[h264 @ 0xae24f4700] Missing reference picture, default is 65568
[h264 @ 0xae24f6d80] mmco: unref short failure


videos: 100%|██████████| 3/3 [2:11:07<00:00, 3358.94s/it]

videos: 100%|██████████| 3/3 [2:11:07<00:00, 2622.35s/it]


9133 frames en /Users/nilsmurallesmorales/universidad/octavosemestre/data/p2/data/processed/frames


In [10]:
# Índice de frames muestreados (cómodo para la Persona 4)
frame_rows = []
for f in FRAMES_DIR.rglob("*.jpg"):
    vid, tpart, label = f.stem.split("__")
    frame_rows.append({"video_id": vid, "time_s": float(tpart[1:-1]),
                       "label": label, "path": str(f.relative_to(PROJECT_DIR))})
if frame_rows:
    fi = pd.DataFrame(frame_rows).sort_values(["video_id", "time_s"])
    fi.to_csv(OUT_DIR / "frames_index.csv", index=False)
    print(fi.shape)
    display(fi["label"].value_counts())

(9133, 4)


label
play         7142
challenge    1051
bg            657
throwin       283
Name: count, dtype: int64

## 8. Salidas

Diccionario de datos completo en **`data/processed/README.md`**.